## Channel Aliasing E2E Test

Validates that `channel_with_alias()` resolves correctly through `development.silver.channel_mapping`,
that `channel()` and `channel_with_alias()` coexist in the same report without errors,
that direct and aliased access to the same physical channel return identical data,
and that persisted `histogram_fact` and `stats_aggregator_fact` are identical
between a direct-channel report and an aliased-channel report.

In [0]:
%pip install lz4

In [0]:
import os
import sys
import numpy as np
from pyspark.sql import functions as F

module_path = os.getcwd()
module_path = "/".join(module_path.split("/")[:-2]) + "/src"
sys.path.insert(0, module_path)

from mda_query_engine.analyze.query.solvers.key_value_store_solver import KeyValueStoreSolver
from mda_query_engine.analyze.query.solvers.solver_config import SolverConfig, TableConfig
from mda_reporting.aggregations.histogram import HistogramDuration
from mda_reporting.aggregations.stats_aggregator import StatsAggregator
from mda_reporting.core.page import Page
from mda_reporting.core.report import Report


In [0]:
SILVER_SCHEMA           = "development.silver"
CONTAINER_METRICS_TABLE = f"{SILVER_SCHEMA}.container_metric"
CHANNEL_METRICS_TABLE   = f"{SILVER_SCHEMA}.channel_metric"
CHANNELS_URI            = f"{SILVER_SCHEMA}.channel_data"
CHANNEL_MAPPING_TABLE   = "development.meta.channel_mapping_v2"

UNITY_CATALOG = "development"
UNITY_SCHEMA  = "gold_e2e"
UNITY_PREFIX  = "alias_e2e"
CONTAINER_TAGS_TABLE = f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{UNITY_PREFIX}_container_tags_tmp"

# Fill these in after running Cell 5 (inspect channel_mapping)
PROJECT_ID          = "HDES"       # value of channel_mapping.project_id
TOOLBOX_ID           = "bev_fleet_monitoring_concept"        # value of channel_mapping.toolbox_id
channel_alias_ENGINE   = "SpdEng"       # e.g. "is1_eng_speed"
DIRECT_CHANNEL_NAME = "is1_eng_speed"     # channel_name in channel_metric for the same physical signal
DIRECT_DATA_KEY     = "TM"
channel_alias_VEH_SPD  = "SpdVeh"    # a second alias for the mixed-mode test (TC-2)


### Setup: Inspect `channel_mapping` and fill in the constants above

In [0]:
mapping_df = spark.read.table(CHANNEL_MAPPING_TABLE)
mapping_df.display()

### Setup: Build a container-id based temporary `container_tags` view

In Databricks, `avl_meta.data_model.concept_entities.entity_id` is not guaranteed to be the numeric `container_id`.
For the HDES data this can contain concept instance identifiers such as `hdes_fmtb`.
`KeyValueStoreSolver` uses `container_tags.entity_id` as the join key into `container_metric` when there are no tag filters,
so the original notebook fails with `CAST_INVALID_INPUT` when Spark tries to compare that string to `container_metric.container_id`.

This notebook only validates channel alias resolution, not tag filtering.
To keep the alias test valid and make the notebook runnable, we build a temporary `container_tags` view directly from `container_metric`
with `entity_id = container_id` and a fixed `project_id` / `toolbox_id` for the current test scope.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UNITY_CATALOG}.{UNITY_SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {CONTAINER_TAGS_TABLE}")

(
    spark.read.table(CONTAINER_METRICS_TABLE)
    .select(
        F.col("container_id").alias("entity_id"),
        F.lit("__identity__").alias("channel_alias"),
        F.lit(PROJECT_ID).alias("project_id"),
        F.lit(TOOLBOX_ID).alias("toolbox_id"),
        F.col("container_id").cast("string").alias("value"),
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(CONTAINER_TAGS_TABLE)
 )

print(f"Temporary container tags table '{CONTAINER_TAGS_TABLE}' created from {CONTAINER_METRICS_TABLE}")
display(spark.read.table(CONTAINER_TAGS_TABLE).limit(5))

### Setup: Reset gold tables for this test prefix

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UNITY_CATALOG}.{UNITY_SCHEMA}")

for tbl in spark.catalog.listTables(f"{UNITY_CATALOG}.{UNITY_SCHEMA}"):
    if tbl.name.startswith(f"{UNITY_PREFIX}_"):
        spark.sql(f"DROP TABLE IF EXISTS {UNITY_CATALOG}.{UNITY_SCHEMA}.{tbl.name}")

print(f"Cleared existing gold tables with prefix '{UNITY_PREFIX}'")

### Shared config builder

In [ ]:
def ensure_alias_container_tags_table() -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UNITY_CATALOG}.{UNITY_SCHEMA}")
    spark.sql(f"DROP TABLE IF EXISTS {CONTAINER_TAGS_TABLE}")

    (
        spark.read.table(CONTAINER_METRICS_TABLE)
        .select(
            F.col("container_id").alias("entity_id"),
            F.lit("__identity__").alias("channel_alias"),
            F.lit(PROJECT_ID).alias("project_id"),
            F.lit(TOOLBOX_ID).alias("toolbox_id"),
            F.col("container_id").cast("string").alias("value"),
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(CONTAINER_TAGS_TABLE)
    )

def build_alias_config(table_prefix: str) -> dict:
    ensure_alias_container_tags_table()
    return {
        "source": {
            "container_tags_table":    CONTAINER_TAGS_TABLE,
            "container_metrics_table": CONTAINER_METRICS_TABLE,
            "channel_metrics_table":   CHANNEL_METRICS_TABLE,
            "channels_uri":            CHANNELS_URI,
            "channel_mapping_table":   CHANNEL_MAPPING_TABLE,
        },
        "unity_sink": {
            "catalog":      UNITY_CATALOG,
            "schema":       UNITY_SCHEMA,
            "table_prefix": table_prefix,
        },
        "query_engine": {
            "solver":         "KeyValueStoreSolver",
            "solver_config": {
                "project_id": PROJECT_ID,
                "container_tags": {
                    "column_name_mapping": {"entity_id": "container_id"},
                },
                "channel_mapping": {
                    "filters": {"toolbox_id": TOOLBOX_ID},
                },
            },
        },
        "measurement_dimensions": ["container_id"],
    }

### TC-1: `channel_with_alias()` returns data; unmatched alias returns empty SampleSeries

In [0]:
solver = KeyValueStoreSolver(
    spark,
    config=SolverConfig(
        project_id=PROJECT_ID,
        container_tags=TableConfig(column_name_mapping={"entity_id": "container_id"}),
        channel_mapping=TableConfig(filters={"toolbox_id": TOOLBOX_ID}),
    ),
)
report_tc1 = Report(name="alias_e2e_tc1", spark=spark, config=build_alias_config(f"{UNITY_PREFIX}_tc1"))
db = report_tc1.get_db()
query = db.query

engine_speed = query.channel_with_alias(channel_alias=channel_alias_ENGINE).alias("engine_speed")
pdf = query.select(engine_speed).toPandas(spark, solver=solver)

assert len(pdf) > 0, "Expected at least one container in result"
assert all(len(row) > 0 for row in pdf["engine_speed"]), \
    "All containers must return non-empty SampleSeries for a valid alias"

# Negative case: unknown alias -> all rows empty
query2 = db.query
no_match = query2.channel_with_alias(channel_alias="__nonexistent_alias__").alias("no_match")
pdf2 = query2.select(no_match).toPandas(spark, solver=solver)
if len(pdf2) > 0:
    assert all(len(row) == 0 for row in pdf2["no_match"]), \
        "Unmatched alias must return SampleSeries.empty() for every container"

print("TC-1 PASSED")


### TC-2: `channel()` and `channel_with_alias()` coexist in the same report

In [0]:
report_tc2 = Report(name="alias_e2e_tc2", spark=spark, config=build_alias_config(f"{UNITY_PREFIX}_tc2"))
query = report_tc2.get_db().query

direct_ch  = query.channel(channel_name=DIRECT_CHANNEL_NAME, data_key=DIRECT_DATA_KEY)
aliased_ch = query.channel_with_alias(channel_alias=channel_alias_VEH_SPD)

page = Page(page_number=1)
report_tc2.add_page(page)
page.add_aggregation(HistogramDuration(
    name="direct_hist",
    base_expr=direct_ch,
    bins=[float(i) for i in range(0, 8000, 250)],
    channel_name=DIRECT_CHANNEL_NAME,
))
page.add_aggregation(HistogramDuration(
    name="alias_hist",
    base_expr=aliased_ch,
    bins=[float(i) for i in range(0, 300, 10)],
    channel_name=channel_alias_VEH_SPD,
))

report_tc2.determine_report()

hist_df_tc2 = report_tc2.aggregation_dfs["HISTOGRAM"]["changed"]
assert hist_df_tc2.filter(F.col("hist_value") > 0).count() > 0, \
    "Expected non-zero hist_value entries after mixed channel + alias report"

print("TC-2 PASSED")

### TC-3: Direct channel and alias of the same physical channel return identical SampleSeries data

In [0]:
solver = KeyValueStoreSolver(
    spark,
    config=SolverConfig(
        project_id=PROJECT_ID,
        container_tags=TableConfig(column_name_mapping={"entity_id": "container_id"}),
        channel_mapping=TableConfig(filters={"toolbox_id": TOOLBOX_ID}),
    ),
)
report_tc3 = Report(name="alias_e2e_tc3", spark=spark, config=build_alias_config(f"{UNITY_PREFIX}_tc3"))
db = report_tc3.get_db()
query = db.query

direct  = query.channel(channel_name=DIRECT_CHANNEL_NAME, data_key=DIRECT_DATA_KEY).alias("direct")
aliased = query.channel_with_alias(channel_alias=channel_alias_ENGINE).alias("aliased")

pdf = query.select(direct, aliased).toPandas(spark, solver=solver)
pdf = pdf.sort_values("container_id").reset_index(drop=True)

mismatches = []
for _, row in pdf.iterrows():
    d = row["direct"]
    a = row["aliased"]
    if len(d) > 0 and len(a) > 0:
        # Both resolved -> values and timestamps must match exactly
        np.testing.assert_array_almost_equal(
            d.values, a.values,
            err_msg=f"Values differ for container_id={row['container_id']}"
        )
        np.testing.assert_array_almost_equal(
            d.tstarts, a.tstarts,
            err_msg=f"Timestamps differ for container_id={row['container_id']}"
        )
    elif len(d) > 0 and len(a) == 0:
        mismatches.append(row["container_id"])

assert len(mismatches) == 0, \
    f"Direct channel returned data but alias returned empty for containers: {mismatches}"

# Alias must resolve for ALL containers, including those with renamed physical channel names
assert all(len(row) > 0 for row in pdf["aliased"]), \
    "Alias must resolve for every container (including those with renamed physical channels)"

print("TC-3 PASSED")


### TC-4: `selector_id` is a deterministic integer, unique per expression

In [0]:
report_tc4 = Report(name="alias_e2e_tc4", spark=spark, config=build_alias_config(f"{UNITY_PREFIX}_tc4"))
db = report_tc4.get_db()
query = db.query

sel_alias_1 = query.channel_with_alias(channel_alias=channel_alias_ENGINE)
sel_alias_2 = query.channel_with_alias(channel_alias=channel_alias_ENGINE)
sel_direct  = query.channel(channel_name=DIRECT_CHANNEL_NAME, data_key=DIRECT_DATA_KEY)

assert isinstance(sel_alias_1.selector_id, int), \
    "selector_id must be an integer"
assert sel_alias_1.selector_id == sel_alias_2.selector_id, \
    "Same expression must always produce the same selector_id (deterministic)"
assert sel_alias_1.selector_id != sel_direct.selector_id, \
    "Different expressions must produce different selector_ids"

print(f"alias selector_id  : {sel_alias_1.selector_id}")
print(f"direct selector_id : {sel_direct.selector_id}")
print("TC-4 PASSED")

### TC-5: `selector_id`s for `channel()` and `channel_with_alias()` of the same channel are merged in `mdf["selector_ids"]`

In [ ]:
solver = KeyValueStoreSolver(
    spark,
    config=SolverConfig(
        project_id=PROJECT_ID,
        container_tags=TableConfig(column_name_mapping={"entity_id": "container_id"}),
        channel_mapping=TableConfig(filters={"toolbox_id": TOOLBOX_ID}),
    ),
)
report_tc5 = Report(name="alias_e2e_tc5", spark=spark, config=build_alias_config(f"{UNITY_PREFIX}_tc5"))
db = report_tc5.get_db()
query = db.query

direct  = query.channel(channel_name=DIRECT_CHANNEL_NAME, data_key=DIRECT_DATA_KEY)
aliased = query.channel_with_alias(channel_alias=channel_alias_ENGINE)

# toPandas triggers cache resolution
pdf = query.select(direct.alias("direct"), aliased.alias("aliased")).toPandas(spark, solver=solver)

# Replicate the solver pipeline to access the merged channels DataFrame
# (selector_ids live in the intermediate channels_df, not on the solver)
direct_selectors = query._collect_time_series_selectors(uses_alias=False)
aliased_selectors = query._collect_time_series_selectors(uses_alias=True)

tags_df = solver.filter_container_tags(spark, query)
metrics_df = solver.filter_container_metrics(spark, query, tags_df)
channel_tags_df = solver.filter_channel_tags(spark, query.db, metrics_df, direct_selectors)

direct_channels = solver.filter_channel_metrics(spark, query.db, channel_tags_df, direct_selectors)
alias_channels = solver.filter_aliased_channel_metrics(spark, query.db, channel_tags_df, aliased_selectors)

mdf = solver.resolve_channel_selections(spark, direct_channels, alias_channels).toPandas()

assert "selector_ids" in mdf.columns, \
    "mdf must contain a selector_ids column after alias solve"

# Every row that the alias resolved must contain aliased.selector_id
for _, row in mdf.iterrows():
    sid_list = row["selector_ids"]
    assert aliased.selector_id in sid_list, \
        f"aliased selector_id missing from mdf row: container_id={row.get('container_id')}"

# For containers where direct also resolved, direct.selector_id must be present too
direct_resolved = mdf[mdf["selector_ids"].apply(lambda s: direct.selector_id in s)]
assert len(direct_resolved) > 0, \
    "direct selector_id should be present for at least some containers"

print("TC-5 PASSED")

### TC-6: Persisted `histogram_fact` and `stats_aggregator_fact` are identical for direct vs. alias

In [0]:
def _add_engine_page(report, channel_expr, channel_name_label):
    """Add a Page with one HistogramDuration and one StatsAggregator for the given channel."""
    page = Page(page_number=1)
    report.add_page(page)
    page.add_aggregation(HistogramDuration(
        name="engine_hist",
        base_expr=channel_expr,
        bins=[float(i) for i in range(0, 8000, 250)],
        channel_name=channel_name_label,
    ))
    page.add_aggregation(StatsAggregator(
        name="engine_stats",
        input_expressions=[channel_expr],
        channel_names=[channel_name_label],
        statistics=["min", "max", "mean"],
    ))


PREFIX_DIRECT = f"{UNITY_PREFIX}_direct"
PREFIX_ALIAS  = f"{UNITY_PREFIX}_alias"

# Clean up both sub-prefixes before running
for prefix in [PREFIX_DIRECT, PREFIX_ALIAS]:
    for tbl in spark.catalog.listTables(f"{UNITY_CATALOG}.{UNITY_SCHEMA}"):
        if tbl.name.startswith(f"{prefix}_"):
            spark.sql(f"DROP TABLE IF EXISTS {UNITY_CATALOG}.{UNITY_SCHEMA}.{tbl.name}")

# Phase A: direct channel report
report_direct = Report(name="alias_e2e_direct", spark=spark, config=build_alias_config(PREFIX_DIRECT))
direct_expr = report_direct.get_db().query.channel(
    channel_name=DIRECT_CHANNEL_NAME, data_key=DIRECT_DATA_KEY
)
_add_engine_page(report_direct, direct_expr, DIRECT_CHANNEL_NAME)
report_direct.determine_report()
report_direct.persist_results()

print("Phase A (direct) persisted")

In [0]:
# Phase B: aliased channel report
report_alias = Report(name="alias_e2e_alias", spark=spark, config=build_alias_config(PREFIX_ALIAS))
aliased_expr = report_alias.get_db().query.channel_with_alias(channel_alias=channel_alias_ENGINE)
_add_engine_page(report_alias, aliased_expr, channel_alias_ENGINE)
report_alias.determine_report()
report_alias.persist_results()

print("Phase B (alias) persisted")

In [0]:
# Phase C: compare persisted fact tables
hist_direct  = spark.read.table(f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{PREFIX_DIRECT}_histogram_fact")
hist_alias   = spark.read.table(f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{PREFIX_ALIAS}_histogram_fact")
stats_direct = spark.read.table(f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{PREFIX_DIRECT}_stats_aggregator_fact")
stats_alias  = spark.read.table(f"{UNITY_CATALOG}.{UNITY_SCHEMA}.{PREFIX_ALIAS}_stats_aggregator_fact")

# Identify containers resolved by BOTH direct and alias (the overlapping set)
direct_cids = {r.container_id for r in hist_direct.select("container_id").distinct().collect()}
alias_cids  = {r.container_id for r in hist_alias.select("container_id").distinct().collect()}
common_cids = list(direct_cids & alias_cids)
assert len(common_cids) > 0, \
    "No common containers between direct and alias reports — check DIRECT_CHANNEL_NAME / channel_alias_ENGINE"
print(f"Comparing {len(common_cids)} common containers: {common_cids}")

# --- Histogram comparison: join on (container_id, bin_id) ---
hist_d = hist_direct.filter(F.col("container_id").isin(common_cids))
hist_a = hist_alias.filter(F.col("container_id").isin(common_cids))

hist_joined = hist_d.alias("d").join(
    hist_a.alias("a"),
    on=["container_id", "bin_id"],
    how="inner",
)
hist_mismatches = hist_joined.filter(
    F.abs(F.col("d.hist_value") - F.col("a.hist_value")) > 1e-9
)
assert hist_mismatches.count() == 0, \
    "histogram_fact hist_value differs between direct and alias report"

# --- Stats comparison: join on (container_id, aggregation_label) ---
stats_d = stats_direct.filter(F.col("container_id").isin(common_cids))
stats_a = stats_alias.filter(F.col("container_id").isin(common_cids))

stats_joined = stats_d.alias("d").join(
    stats_a.alias("a"),
    on=["container_id", "aggregation_label"],
    how="inner",
)
stats_mismatches = stats_joined.filter(
    F.abs(F.col("d.statistic_value") - F.col("a.statistic_value")) > 1e-9
)
assert stats_mismatches.count() == 0, \
    "stats_aggregator_fact statistic_value differs between direct and alias report"

print("TC-6 PASSED — histogram_fact and stats_aggregator_fact are identical for common containers")

In [0]:
print("=" * 60)
print("CHANNEL ALIASING E2E — ALL TESTS PASSED")
print("=" * 60)
print("  TC-1: channel_with_alias() returns data / empty SampleSeries")
print("  TC-2: channel() + channel_with_alias() coexist in same report")
print("  TC-3: direct and alias return identical SampleSeries data")
print("  TC-4: selector_id is deterministic and expression-unique")
print("  TC-5: selector_ids for direct + alias merged in mdf row")
print("  TC-6: histogram_fact and stats_aggregator_fact are identical")